In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

True

In [3]:
from langchain_neo4j import Neo4jGraph

graph = Neo4jGraph(
    url=os.getenv("NEO4J_URI"),
    username=os.getenv("NEO4J_USERNAME"),  
    password=os.getenv("NEO4J_PASSWORD"),
) 

In [4]:
# 테스트 쿼리 실행 
cypher_query = """
CREATE (n:Test {name: "Hello AuraDB"}) 
RETURN n
"""

graph.query(cypher_query)

[{'n': {'name': 'Hello AuraDB'}}]

In [5]:
def reset_database(graph):
    """
    데이터베이스 초기화하기
    """
    # 모든 노드와 관계 삭제
    graph.query("MATCH (n) DETACH DELETE n")
    
    # 모든 제약조건 삭제
    constraints = graph.query("SHOW CONSTRAINTS")
    for constraint in constraints:
        constraint_name = constraint.get("name")
        if constraint_name:
            graph.query(f"DROP CONSTRAINT {constraint_name}")
    
    # 모든 인덱스 삭제
    indexes = graph.query("SHOW INDEXES")
    for index in indexes:
        index_name = index.get("name")
        index_type = index.get("type")
        if index_name and index_type != "CONSTRAINT":
            graph.query(f"DROP INDEX {index_name}")
    
    print("데이터베이스가 초기화되었습니다.")

# 데이터베이스 초기화
reset_database(graph)

데이터베이스가 초기화되었습니다.


In [6]:
# 데이터 모델 정의 
cypher_query =  """
// Person 노드 생성
CREATE (p1:Person {name: '홍길동', age: 35, city: '서울'})
CREATE (p2:Person {name: '김철수', age: 28, city: '부산'})
CREATE (p3:Person {name: '이영희', age: 32, city: '서울'})
CREATE (p4:Person {name: '박지성', age: 40, city: '인천'})
CREATE (p5:Person {name: '최민수', age: 45, city: '대전'})
CREATE (p6:Person {name: '정소라', age: 29, city: '부산'})
CREATE (p7:Person {name: '강대호', age: 33, city: '서울'})
CREATE (p8:Person {name: '윤미래', age: 27, city: '대구'})

// 관심분야 노드 생성
CREATE (i1:Interest {name: '영화'})
CREATE (i2:Interest {name: '음악'})
CREATE (i3:Interest {name: '스포츠'})
CREATE (i4:Interest {name: '요리'})
CREATE (i5:Interest {name: '여행'})

// 장소 노드 생성
CREATE (l1:Location {name: '남산타워', city: '서울'})
CREATE (l2:Location {name: '해운대', city: '부산'})
CREATE (l3:Location {name: '경복궁', city: '서울'})
CREATE (l4:Location {name: '월미도', city: '인천'})

// KNOWS 관계 생성 (사람들 간의 친구 관계)
CREATE (p1)-[:KNOWS {since: 2010}]->(p2)
CREATE (p1)-[:KNOWS {since: 2015}]->(p3)
CREATE (p1)-[:KNOWS {since: 2012}]->(p7)
CREATE (p2)-[:KNOWS {since: 2018}]->(p6)
CREATE (p3)-[:KNOWS {since: 2017}]->(p4)
CREATE (p3)-[:KNOWS {since: 2020}]->(p5)
CREATE (p4)-[:KNOWS {since: 2019}]->(p5)
CREATE (p6)-[:KNOWS {since: 2016}]->(p8)
CREATE (p7)-[:KNOWS {since: 2014}]->(p3)
CREATE (p7)-[:KNOWS {since: 2021}]->(p8)

// LIKES 관계 생성 (사람과 관심분야 간의 관계)
CREATE (p1)-[:LIKES {rating: 5}]->(i1)
CREATE (p1)-[:LIKES {rating: 4}]->(i5)
CREATE (p2)-[:LIKES {rating: 5}]->(i3)
CREATE (p3)-[:LIKES {rating: 4}]->(i2)
CREATE (p3)-[:LIKES {rating: 5}]->(i4)
CREATE (p4)-[:LIKES {rating: 5}]->(i3)
CREATE (p5)-[:LIKES {rating: 3}]->(i1)
CREATE (p5)-[:LIKES {rating: 4}]->(i5)
CREATE (p6)-[:LIKES {rating: 5}]->(i2)
CREATE (p7)-[:LIKES {rating: 4}]->(i4)
CREATE (p8)-[:LIKES {rating: 4}]->(i5)

// VISITED 관계 생성 (사람과 장소 간의 관계)
CREATE (p1)-[:VISITED {date: '2022-03-15', rating: 4}]->(l1)
CREATE (p1)-[:VISITED {date: '2022-05-20', rating: 5}]->(l2)
CREATE (p2)-[:VISITED {date: '2022-04-10', rating: 3}]->(l2)
CREATE (p3)-[:VISITED {date: '2022-06-05', rating: 5}]->(l3)
CREATE (p4)-[:VISITED {date: '2022-07-12', rating: 4}]->(l4)
CREATE (p4)-[:VISITED {date: '2022-02-28', rating: 5}]->(l1)
CREATE (p5)-[:VISITED {date: '2022-08-03', rating: 4}]->(l3)
CREATE (p6)-[:VISITED {date: '2022-01-17', rating: 5}]->(l2)
CREATE (p7)-[:VISITED {date: '2022-09-22', rating: 3}]->(l1)
CREATE (p8)-[:VISITED {date: '2022-10-08', rating: 4}]->(l2)
"""

graph.query(cypher_query)

[]

In [ ]:
cypher_query = """
MATCH (p1:Person{name:'홍길동'})-[r:KNOWS*1..3]->(p2:Person)
RETURN p1.name AS start,[rel in r | type(rel) + '('+ rel.since + ')'] AS relationships, p2.name AS end

"""
graph.query(cypher_query)


[{'start': '홍길동', 'relationships': ['KNOWS(2010)'], 'end': '김철수'},
 {'start': '홍길동',
  'relationships': ['KNOWS(2010)', 'KNOWS(2018)'],
  'end': '정소라'},
 {'start': '홍길동',
  'relationships': ['KNOWS(2010)', 'KNOWS(2018)', 'KNOWS(2016)'],
  'end': '윤미래'},
 {'start': '홍길동', 'relationships': ['KNOWS(2015)'], 'end': '이영희'},
 {'start': '홍길동',
  'relationships': ['KNOWS(2015)', 'KNOWS(2017)'],
  'end': '박지성'},
 {'start': '홍길동',
  'relationships': ['KNOWS(2015)', 'KNOWS(2017)', 'KNOWS(2019)'],
  'end': '최민수'},
 {'start': '홍길동',
  'relationships': ['KNOWS(2015)', 'KNOWS(2020)'],
  'end': '최민수'},
 {'start': '홍길동', 'relationships': ['KNOWS(2012)'], 'end': '강대호'},
 {'start': '홍길동',
  'relationships': ['KNOWS(2012)', 'KNOWS(2014)'],
  'end': '이영희'},
 {'start': '홍길동',
  'relationships': ['KNOWS(2012)', 'KNOWS(2014)', 'KNOWS(2017)'],
  'end': '박지성'},
 {'start': '홍길동',
  'relationships': ['KNOWS(2012)', 'KNOWS(2014)', 'KNOWS(2020)'],
  'end': '최민수'},
 {'start': '홍길동',
  'relationships': ['KNOWS(2012)'

In [12]:
cypher_query = """
MATCH (p1:Person{name:'홍길동'})-[r:KNOWS*2]->(p2:Person)
RETURN p1.name AS start, p2.name AS end

"""
graph.query(cypher_query)

[{'start': '홍길동', 'end': '이영희'},
 {'start': '홍길동', 'end': '박지성'},
 {'start': '홍길동', 'end': '최민수'},
 {'start': '홍길동', 'end': '정소라'},
 {'start': '홍길동', 'end': '윤미래'}]

In [13]:
cypher_query = """

SyntaxError: incomplete input (2383436071.py, line 1)

In [ ]:
cypher_query = """
MATCH (p:Person{name: '홍길동'})-[r:KNOWS*]->(p2:Person)
RETURN p.name AS start, p2.name AS end, SIZE(r) as degree
"""
graph.query(cypher_query)

[{'start': '홍길동', 'end': '김철수', 'degree': 1},
 {'start': '홍길동', 'end': '정소라', 'degree': 2},
 {'start': '홍길동', 'end': '윤미래', 'degree': 3},
 {'start': '홍길동', 'end': '이영희', 'degree': 1},
 {'start': '홍길동', 'end': '박지성', 'degree': 2},
 {'start': '홍길동', 'end': '최민수', 'degree': 3},
 {'start': '홍길동', 'end': '최민수', 'degree': 2},
 {'start': '홍길동', 'end': '강대호', 'degree': 1},
 {'start': '홍길동', 'end': '이영희', 'degree': 2},
 {'start': '홍길동', 'end': '박지성', 'degree': 3},
 {'start': '홍길동', 'end': '최민수', 'degree': 4},
 {'start': '홍길동', 'end': '최민수', 'degree': 3},
 {'start': '홍길동', 'end': '윤미래', 'degree': 2}]

: 

In [4]:
#shortestPath 쿼리
cypher_query = """
MATCH p=shortestPath((p1:Person{name:'홍길동'})-[:KNOWS*]-(p2:Person{name:'최민수'}))
return [node in nodes(p) | node.name] AS path;
"""
graph.query(cypher_query)

[{'path': ['홍길동', '이영희', '최민수']}]

In [ ]:
#shortestPath 쿼리
cypher_query = """
MATCH p=shortestPath((p1:Person{name:'홍길동'})-[*..3]-(p2:Person{name:'정소라'}))
return [node in nodes(p) | node.name] AS path;
"""
graph.query(cypher_query)

[{'path': ['홍길동', '김철수', '정소라']}]

In [9]:
cypher_query = """
MATCH (p1:Person)
return count(p1) as total_people
"""
graph.query(cypher_query)

[{'total_people': 8}]

In [11]:
cypher_query = """
MATCH (p:Person)
RETURN p.city as city, count(p) as total_people
ORDER BY total_people DESC
"""
graph.query(cypher_query)

[{'city': '서울', 'total_people': 3},
 {'city': '부산', 'total_people': 2},
 {'city': '인천', 'total_people': 1},
 {'city': '대전', 'total_people': 1},
 {'city': '대구', 'total_people': 1}]

In [14]:
cypher_query = """
Match (p:Person)
return  AVG(p.age) as average_age
"""
graph.query(cypher_query)

[{'average_age': 33.62500000000001}]

In [ ]:
cypher_query = """
Match (p:Person)-[:LIKES]->(i:Interest)
RETURN i.name as interest, count(p) as total_people
ORDER BY total_people DESC
"""
graph.query(cypher_query)

[{'interest': '여행', 'total_people': 3},
 {'interest': '영화', 'total_people': 2},
 {'interest': '스포츠', 'total_people': 2},
 {'interest': '음악', 'total_people': 2},
 {'interest': '요리', 'total_people': 2}]

In [21]:
cypher_query = """
match (p:Person)-[:VISITED]->(l:Location)
RETURN l.name as location, count(p) as total_people
order by total_people DESC
"""
graph.query(cypher_query)

[{'location': '해운대', 'total_people': 4},
 {'location': '남산타워', 'total_people': 3},
 {'location': '경복궁', 'total_people': 2},
 {'location': '월미도', 'total_people': 1}]

In [22]:
cypher_query = """
match (p:Person)-[:VISITED]->(l:Location)
RETURN l.name as location, avg(l.rating) as average_rating
order by average_rating DESC
limit 1
"""
graph.query(cypher_query)

[{'location': '남산타워', 'average_rating': None}]